In [1]:
import torch
import torch.nn as nn

In [2]:
#create model class
## here out model has single percenptron with certain number of features 
### it uses sigmoid activation function
class Model(nn.Module):
    def __init__ (self, num_features):
        super().__init__()

        self.linear= nn.Linear(in_features= num_features, out_features= 1)
        self.sigmoid= nn.Sigmoid()

    def forward( self, features):
        out= self.linear(features)
        out= self.sigmoid(out)

        return out

In [3]:
# create dataset
features= torch.rand(size= (10, 5))

In [4]:
# create a model
model= Model(features.shape[1])

In [5]:
# call model for forward pass
## we can do model.forward(features) but it isnt recommended by pytorch
model(features)

tensor([[0.5407],
        [0.6616],
        [0.5513],
        [0.5252],
        [0.6130],
        [0.5625],
        [0.6445],
        [0.6170],
        [0.5893],
        [0.5496]], grad_fn=<SigmoidBackward0>)

In [6]:
## to get the weights going to each layers
model.linear.weight

Parameter containing:
tensor([[ 0.2183,  0.3515, -0.1856,  0.4462,  0.3208]], requires_grad=True)

In [7]:
model.linear.bias

Parameter containing:
tensor([-0.2924], requires_grad=True)

In [8]:
# to visualize our model
!pip install torchinfo

In [9]:
from torchinfo import summary

summary(model, input_size= (10,5))

Layer (type:depth-idx)                   Output Shape              Param #
Model                                    [10, 1]                   --
├─Linear: 1-1                            [10, 1]                   6
├─Sigmoid: 1-2                           [10, 1]                   --
Total params: 6
Trainable params: 6
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

## Making more complex NN
- Here we will have hidden layer with 3 neurons and a output layer with single neuron
- Hidden layer uses Relu whereas output layer uses sigmoid
- we will have 5 inputs(input columns)

In [19]:
class MyNN(nn.Module):
    def __init__(self, num_features):

        super().__init__()

        '''
        self.linear1= nn.Linear(in_features= num_features, out_features= 3)
        self.relu= nn.ReLU()

        self.linear2= nn.Linear(3, 1)
        self.sigmoid= nn.Sigmoid()

        '''
        # we can also make sequential containers here so that we donot have to manually send output of one layer as ip to another layer


        self.network= nn.Sequential(
            # 1st hidden layer
            nn.Linear(in_features= num_features, out_features= 3),
            nn.ReLU(),

            # output layer
            nn.Linear(3, 1),
            nn.Sigmoid(),

        )

    def forward(self, features):
        """
        out= self.linear1(features)
        out= self.relu(out)

        out= self.linear2(out)
        out= self.sigmoid(out)


        Instead of this above we can simply do this now
        """

        out= self.network(features)


        return out


In [20]:
model2= MyNN(features.shape[1])



In [21]:
model2(features)

tensor([[0.4590],
        [0.4619],
        [0.4486],
        [0.4502],
        [0.4708],
        [0.4484],
        [0.4661],
        [0.4501],
        [0.4481],
        [0.4493]], grad_fn=<SigmoidBackward0>)

In [24]:
model2.network[0].weight

Parameter containing:
tensor([[-0.3686,  0.2143, -0.2845,  0.2303, -0.0797],
        [-0.4173,  0.2320,  0.3136, -0.2483, -0.2233],
        [-0.1541, -0.3203, -0.3591,  0.3555, -0.2367]], requires_grad=True)

In [27]:
# model2.linear2.weight
model2.network[2].weight

Parameter containing:
tensor([[ 0.4846, -0.1073,  0.2572]], requires_grad=True)

In [29]:
model2.network[0].bias

Parameter containing:
tensor([ 0.0246,  0.3490, -0.2853], requires_grad=True)

In [28]:
model2.network[2].bias

Parameter containing:
tensor([-0.1800], requires_grad=True)

In [30]:
summary(model2, input_size=(10, 5))

Layer (type:depth-idx)                   Output Shape              Param #
MyNN                                     [10, 1]                   --
├─Sequential: 1-1                        [10, 1]                   --
│    └─Linear: 2-1                       [10, 3]                   18
│    └─ReLU: 2-2                         [10, 3]                   --
│    └─Linear: 2-3                       [10, 1]                   4
│    └─Sigmoid: 2-4                      [10, 1]                   --
Total params: 22
Trainable params: 22
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

## Training precious manual code using Pytorch

In [50]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split


In [51]:
data= load_breast_cancer()

In [52]:
x= data['data']
y= data['target']

In [53]:
x_train, x_test, y_train, y_test= train_test_split(x, y, random_state= 42, test_size= 0.2)

In [54]:
scaler= StandardScaler()
x_train= scaler.fit_transform(x_train)
x_test= scaler.fit_transform(x_test)

In [55]:
x_train_tensor= torch.from_numpy(x_train).float()
x_test_tensor= torch.from_numpy(x_test).float()
y_train_tensor= torch.from_numpy(y_train).float()
y_test_tensor= torch.from_numpy(y_test).float()

In [72]:
# making our model
class ClassificationModel(nn.Module):
    def __init__ (self, num_features):

        super(). __init__()

        self.linear= nn.Linear(num_features, 1)
        self.sigmoid= nn.Sigmoid()



    def forward(self, features):
        out= self.linear(features)
        out= self.sigmoid(out)

        return out

In [73]:
# making instance of our model

classmodel= ClassificationModel(x_train_tensor.shape[1])


In [80]:
# making optimizer
optimizer= torch.optim.SGD(classmodel.parameters(), lr= 0.1)



In [81]:
#loss function
loss_function= nn.BCELoss()

In [82]:
# making loop
for i in range(25):


    #clears previous gradient of all params
    optimizer.zero_grad()

    #forward propagation
    y_pred= classmodel(x_train_tensor)   # if we pass only one row then automatically behaves as sgd


    #calculate loss
    loss= loss_function(y_pred, y_train_tensor.reshape(-1, 1))


    #claculate gradient
    loss.backward()

    #update params
    optimizer.step()

    print(f"Epoch {i+1}; Loss: {loss}")


    


Epoch 1; Loss: 0.5639942288398743
Epoch 2; Loss: 0.4624215364456177
Epoch 3; Loss: 0.40045174956321716
Epoch 4; Loss: 0.35850024223327637
Epoch 5; Loss: 0.32799869775772095
Epoch 6; Loss: 0.30466899275779724
Epoch 7; Loss: 0.28614234924316406
Epoch 8; Loss: 0.27100029587745667
Epoch 9; Loss: 0.25834035873413086
Epoch 10; Loss: 0.24756014347076416
Epoch 11; Loss: 0.23824144899845123
Epoch 12; Loss: 0.23008431494235992
Epoch 13; Loss: 0.22286781668663025
Epoch 14; Loss: 0.2164251208305359
Epoch 15; Loss: 0.21062786877155304
Epoch 16; Loss: 0.2053755223751068
Epoch 17; Loss: 0.20058803260326385
Epoch 18; Loss: 0.19620086252689362
Epoch 19; Loss: 0.19216130673885345
Epoch 20; Loss: 0.18842579424381256
Epoch 21; Loss: 0.1849581003189087
Epoch 22; Loss: 0.18172767758369446
Epoch 23; Loss: 0.17870862782001495
Epoch 24; Loss: 0.1758788824081421
Epoch 25; Loss: 0.1732194572687149


In [84]:
with torch.no_grad():
    y_test_pred= classmodel(x_test_tensor)

    

In [87]:
y_test_pred= (y_test_pred>= 0.5).float()
y_test_pred[:5]

tensor([[1.],
        [0.],
        [0.],
        [1.],
        [1.]])

In [88]:
accuracy= (y_test_pred==y_test_tensor).float().mean()
accuracy.item()

0.5323176383972168